In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("winequality-red.csv" ,sep=";")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
df["quality"].value_counts()

quality
5    681
6    638
7    199
4     53
8     18
3     10
Name: count, dtype: int64

In [4]:
df["quality_binary"]=df["quality"].apply(lambda x:1 if x>=6 else 0)

In [5]:
x=df.drop(["quality","quality_binary"],axis=1)

In [6]:
y=df["quality_binary"]

In [7]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [8]:
x_train.shape,y_train.shape

((1279, 11), (1279,))

In [9]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [10]:
metamodel=LogisticRegression()

In [11]:
base_learner=[
    ("dt",DecisionTreeClassifier(random_state=42)),
    ("knn",KNeighborsClassifier()),
    ("svm",SVC(probability=True,random_state=42))
]

In [12]:
stack_model=StackingClassifier(
    estimators=base_learner,
    final_estimator=metamodel,
    cv=5
)

In [13]:
stack_model.fit(x_train_scaled,y_train)

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('knn', KNeighborsClassifier()),
                               ('svm', SVC(probability=True, random_state=42))],
                   final_estimator=LogisticRegression())

In [14]:
y_pred=stack_model.predict(x_test_scaled)


In [15]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.73      0.77      0.75       149
           1       0.79      0.75      0.77       171

    accuracy                           0.76       320
   macro avg       0.76      0.76      0.76       320
weighted avg       0.76      0.76      0.76       320



In [16]:
accuracy_score(y_test,y_pred)

0.75625